# **Import Libraries**

In [1]:
import pandas as pd
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# **Download data**

In [2]:
df=pd.read_csv("/kaggle/input/datasets/ulrikthygepedersen/online-retail-dataset/online_retail.csv")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


# **Data cleaning**

In [5]:
df = df.dropna(subset=['CustomerID'])
df.isnull().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

In [6]:
df['CustomerID'] = df['CustomerID'].astype(int)


In [7]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 406829 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    406829 non-null  object        
 1   StockCode    406829 non-null  object        
 2   Description  406829 non-null  object        
 3   Quantity     406829 non-null  int64         
 4   InvoiceDate  406829 non-null  datetime64[ns]
 5   UnitPrice    406829 non-null  float64       
 6   CustomerID   406829 non-null  int64         
 7   Country      406829 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 27.9+ MB


In [9]:
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

In [10]:
df['Total_Amount'] = df['Quantity'] * df['UnitPrice']
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Total_Amount
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


# **Outliers**

In [11]:
customer_revenue = df.groupby('CustomerID')['Total_Amount'].sum()
print(customer_revenue.describe())
print("\n--- Top 5 Customers by Revenue ---")
print(customer_revenue.nlargest(5))

count      4338.000000
mean       2054.266460
std        8989.230441
min           3.750000
25%         307.415000
50%         674.485000
75%        1661.740000
max      280206.020000
Name: Total_Amount, dtype: float64

--- Top 5 Customers by Revenue ---
CustomerID
14646    280206.02
18102    259657.30
17450    194550.79
16446    168472.50
14911    143825.06
Name: Total_Amount, dtype: float64


## Business Context: Wholesale Buyers vs. Retail Customers

The revenue distribution above shows a large gap between the mean (2054) and the 
median (674), along with a handful of customers spending 100x+ the typical amount. 
This is consistent with the nature of the Online Retail dataset, which includes 
both individual consumers and wholesale/B2B buyers purchasing in bulk.

These extreme spenders aren't removed from the analysis, since they represent real, 
valuable customers — but it's worth quantifying how concentrated the revenue is 
before drawing conclusions from averages.

In [12]:
# What share of total revenue comes from the top 1% of customers?
top_1_pct_count = max(1, int(len(customer_revenue) * 0.01))
top_1_pct_revenue = customer_revenue.nlargest(top_1_pct_count).sum()
total_revenue_check = customer_revenue.sum()

print(f"Top {top_1_pct_count} customers ({top_1_pct_count/len(customer_revenue)*100:.1f}% of all customers)")
print(f"account for {top_1_pct_revenue/total_revenue_check*100:.1f}% of total revenue.")

Top 43 customers (1.0% of all customers)
account for 31.8% of total revenue.


# **Building an RFM table**

In [13]:
snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)


rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency
    'InvoiceNo': 'nunique',                                   # Frequency
    'Total_Amount': 'sum'                                     # Monetary
})


rfm.rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',
    'Total_Amount': 'Monetary'
}, inplace=True)

print(rfm.head())

            Recency  Frequency  Monetary
CustomerID                              
12346           326          1  77183.60
12347             2          7   4310.00
12348            75          4   1797.24
12349            19          1   1757.55
12350           310          1    334.40


# **R/F/M Scores**

In [14]:
r_labels = range(5, 0, -1)
f_labels = range(1, 6)
m_labels = range(1, 6)

rfm['R'] = pd.qcut(rfm['Recency'].rank(method='first'), q=5, labels=r_labels)
rfm['F'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=f_labels)
rfm['M'] = pd.qcut(rfm['Monetary'].rank(method='first'), q=5, labels=m_labels)

rfm['RFM_Segment'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)
rfm.head()

,Recency,Frequency,Monetary,R,F,M,RFM_Segment
CustomerID,,,,,,,
12346,326,1,77183.60,1,1,5,115
12347,2,7,4310.00,5,5,5,555
12348,75,4,1797.24,2,4,4,244
12349,19,1,1757.55,4,1,4,414
12350,310,1,334.40,1,1,2,112


# **Customer Segmentation**

In [15]:
def segment_me(df):
    r = int(df['R'])
    f = int(df['F'])

    if r >= 4 and f >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 3 and f <= 2:
        return 'Promising / New'
    elif r <= 2 and f >= 3:
        return 'At Risk'
    else:
        return 'Lost Customers'

rfm['Customer_Segment'] = rfm.apply(segment_me, axis=1)
print(rfm['Customer_Segment'].value_counts())

Customer_Segment
Champions          1121
Lost Customers     1074
Loyal Customers     821
At Risk             661
Promising / New     661
Name: count, dtype: int64


In [16]:
segment_summary = rfm.groupby('Customer_Segment').agg(
    Customer_Count=('Recency', 'count'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Monetary=('Monetary', 'mean'),
    Total_Revenue=('Monetary', 'sum')
).round(1).sort_values('Total_Revenue', ascending=False)

segment_summary

,Customer_Count,Avg_Recency,Avg_Frequency,Avg_Monetary,Total_Revenue
Customer_Segment,,,,,
Champions,1121,13.0,10.1,5238.1,5871961.5
Loyal Customers,821,37.2,3.7,1687.7,1385600.3
At Risk,661,150.6,3.4,1250.3,826438.4
Lost Customers,1074,216.7,1.1,488.7,524889.7
Promising / New,661,36.4,1.2,457.7,302518.0


# **Pareto Insight**

In [17]:
total_customers = rfm.shape[0]
total_revenue = rfm['Monetary'].sum()

pareto = segment_summary.copy()
pareto['Customer_%'] = (pareto['Customer_Count'] / total_customers * 100).round(1)
pareto['Revenue_%'] = (pareto['Total_Revenue'] / total_revenue * 100).round(1)

print(pareto[['Customer_%', 'Revenue_%']])

champions_row = pareto.loc['Champions']
print(f"\nChampions represent {champions_row['Customer_%']}% of customers"
      f" but generate {champions_row['Revenue_%']}% of total revenue.")

                  Customer_%  Revenue_%
Customer_Segment                       
Champions               25.8       65.9
Loyal Customers         18.9       15.5
At Risk                 15.2        9.3
Lost Customers          24.8        5.9
Promising / New         15.2        3.4

Champions represent 25.8% of customers but generate 65.9% of total revenue.


# **Visualization**

In [18]:
segment_order = rfm['Customer_Segment'].value_counts().index

fig = px.bar(
    rfm['Customer_Segment'].value_counts().reindex(segment_order).reset_index(),
    x='Customer_Segment',
    y='count',
    text='count',
    color='Customer_Segment',
    color_discrete_sequence=px.colors.sequential.Blues_r,
    title='Customer Segments Distribution'
)

fig.update_traces(textposition='outside')
fig.update_layout(
    xaxis_title='Customer Segment',
    yaxis_title='Number of Customers',
    showlegend=False,
    template='plotly_white'
)
fig.show()

In [19]:
customer_counts = rfm.groupby('Customer_Segment')['Recency'].count().sort_values(ascending=False)

fig = px.pie(
    values=customer_counts.values,
    names=customer_counts.index,
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Set2,
    title='Customer Count Breakdown (%)'
)

fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(template='plotly_white')
fig.show()

In [20]:
revenue_counts = rfm.groupby('Customer_Segment')['Monetary'].sum().sort_values(ascending=False)

fig = px.pie(
    values=revenue_counts.values,
    names=revenue_counts.index,
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Set2,
    title='Total Revenue Breakdown (%)'
)

fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(template='plotly_white')
fig.show()

In [21]:
heatmap_data = rfm.pivot_table(index='R', columns='F', values='Monetary', aggfunc='mean')

fig = px.imshow(
    heatmap_data,
    text_auto='.0f',
    color_continuous_scale='YlOrRd',
    labels=dict(x='Frequency Score (F)', y='Recency Score (R)', color='Avg Monetary'),
    title='Average Monetary Value by Recency & Frequency Score'
)

fig.update_layout(template='plotly_white')
fig.show()

/tmp/ipykernel_16/2666211706.py:1: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior



# **Business Recommendations**

The segment distribution follows a clear Pareto pattern: **Champions represent 
25.8% of customers but generate 65.9% of total revenue** — confirming that a 
relatively small group drives the majority of the business.

| Segment | % of Customers | % of Revenue | Recommended Action | Priority |
|---|---|---|---|---|
| **Champions** | 25.8% | 65.9% | This is where the business lives. VIP programs, early access to new products, personal account manager, ask for referrals. Avoid heavy discount campaigns — they don't buy because of price, and discounts can erode brand perception with this group. | Critical (retain at all costs) |
| **Loyal Customers** | 18.9% | 15.5% | Second-largest revenue contributor. Loyalty/points programs to push them toward Champions, cross-sell and up-sell based on purchase history, use them for feedback surveys. | High |
| **At Risk** | 15.2% | 9.3% | These were historically frequent buyers who've gone quiet — high potential value if reactivated. Personalized win-back campaigns, direct outreach rather than generic bulk emails. | High (time-sensitive) |
| **Promising / New** | 15.2% | 3.4% | Low current revenue but represents future Champions. Strong onboarding experience, second-purchase discount (not the first), timely follow-up after the first order. | Medium |
| **Lost Customers** | 24.8% | 5.9% | Largest customer count but lowest revenue share — recovery cost usually exceeds expected value. One final "best offer" campaign, then analyze churn reasons instead of repeated re-engagement attempts. | Low |

### Project-level Recommendations

1. **Protect the 25.8%** — since Champions alone generate two-thirds of revenue, 
   even a small increase in churn within this segment would have an outsized 
   impact on total revenue. Retention efforts here should take priority over 
   acquisition spend.
2. **Re-run RFM periodically** (every 3–6 months) — customers move between 
   segments over time, so this is not a one-time analysis.
3. **Enrich RFM with additional data** in the next phase, such as product 
   affinity or preferred purchase channel, for more precise targeting.
4. **A/B test the recommended campaigns** before rolling them out permanently 
   (e.g. a 10% vs. 20% win-back discount for the At Risk segment).
5. **Limitations** — RFM is based purely on historical transactional behavior 
   and does not capture the *reasons* behind that behavior (seasonality, 
   external factors, etc.). It should be treated as a starting point for 
   targeting, not a final decision on its own.